# Carderno 5 - Gerar embeddings dos documentos e queries

In [1]:
import json
from openai import OpenAI
import os
from tqdm import tqdm
from getpass import getpass
import h5py
import numpy as np
from formatador import html_to_plain_text
import re
import faiss

In [2]:
PASTA_DOCS_QRELS = './dados/outputs/0 - qrel - docs - query - raw_human_eval/'

ARQUIVO_DOCS = f'{PASTA_DOCS_QRELS}/docs.csv'
ARQUIVO_QUERIES = f'{PASTA_DOCS_QRELS}/query.csv'
ARQUIVO_QRELS = f'{PASTA_DOCS_QRELS}/qrel.csv'

In [3]:
OPENAI_KEY = getpass("KEY OpenAI")
NOME_MODELO_EMB_OPENAI_LARGE = "text-embedding-3-large"
DIM_MODELO_EMB_OPENAI_LARGE = 3072

NOME_MODELO_EMB_OPENAI_SMALL = "text-embedding-3-small"
DIM_MODELO_EMB_OPENAI_SMALL = 1536 
# Modelos disponíveis
MODELOS_EMB_NOME_E_DIM_EMB = [(NOME_MODELO_EMB_OPENAI_LARGE, DIM_MODELO_EMB_OPENAI_LARGE), (NOME_MODELO_EMB_OPENAI_SMALL, DIM_MODELO_EMB_OPENAI_SMALL)]

# Modelos para gerar. A ideia é que, uma vez que já foi gerado, pode tirar daqui. Daí ele não precisa carregar o arquivo e ver se está lá
MODELOS_EMB_PARA_GERAR = [(NOME_MODELO_EMB_OPENAI_SMALL, DIM_MODELO_EMB_OPENAI_SMALL)]
MODELOS_EMB_PARA_GERAR = []

KEY OpenAI ········


Os embeddings serão gerados no arquivo abaixo. No entanto, depois vou copiar um conjunto de embeddings por arquivo para subir pro git.

In [4]:
ARQUIVO_EMBEDDINGS_DOCS = './dados/outputs/5 - embeddings/embeddings_docs.h5'
ARQUIVO_EMBEDDINGS_QUERIES = './dados/outputs/5 - embeddings/embeddings_queries.h5'

# 1. Carregar a base de documentos e de queries

In [5]:
import pandas as pd

docs = pd.read_csv(ARQUIVO_DOCS)
queries = pd.read_csv(ARQUIVO_QUERIES)
qrels = pd.read_csv(ARQUIVO_QRELS)

docs['TEXTONORMA'] = docs['TEXTONORMA'].fillna('')
docs['ASSUNTO'] = docs['ASSUNTO'].fillna('')

In [6]:
print(docs.columns)
print(queries.columns)

Index(['KEY', 'UNIDADEBASICAAUTORA', 'ORIGEM', 'NUMNORMA', 'ANONORMA',
       'TIPONORMA', 'NUMEROPROCESSO', 'NUMEROPROCESSOFORMATADO', 'TITULO',
       'ASSUNTO', 'TEXTONORMA', 'DATAINICIOVIGENCIA', 'DATAFIMVIGENCIA',
       'SITUACAO', 'LINKBTCU', 'TEXTOANEXO', 'ARQUIVONORMA', 'PAGINABTCU',
       'TEMA', 'TAGSVCE', 'NORMARELACIONADA', 'NUMDOU', 'NUMSECAODOU',
       'NUMPAGINADOU', 'DATADOU', 'INFOSGERAIS'],
      dtype='object')
Index(['KEY', 'TEXT'], dtype='object')


# 2. Criar as estruturas em arquivos H5

In [7]:
# Cria uma estrutura h5. A estrutura é salva no arquivo 'arquivo'.
# A ideia da estrutura é a seguinte:
#   - Um dataset contendo a lista de ids que será salvo. 
#       O nome desse dataset poderia ser padrão (por exemplo, ID) já que será um h5 para documentos e outro para as queries. No entanto,
#       usarei DOC_KEY e QUERY_KEY simplesmente pq é o vocabulário que estamos usando para docs e queries. Esse dataset já é inicializado
#       com a lista de todas as ids.
#   - Um dataset por modelo de embeddings. Nesse caso, o nome do dataset é o nome do modelo de embeddings e, quando é criado, é criado
#       sem embeddings (serão gerados depois). A ideia é que os embeddings são pareados com as ids, ou seja, o embeddings na posição i
#       se refere ao i'éssima id.
def criar_estrutura_embeddings(arquivo, nome_id, lista_id):
    with h5py.File(arquivo, "a") as f:
        # Tamanho do dataset
        n = len(lista_id)
        # Tamanho dos chunks para gravar no dataset
        chunk_size = min(128, n)
        
        # Cria o dataset para as IDs dos docs/queries
        if nome_id not in f:
            f.create_dataset(
                nome_id,
                data=np.array(lista_id, dtype="S"),  # grava tudo de uma vez
                maxshape=(None,),
                dtype=h5py.string_dtype(encoding="utf-8"),
                chunks=True
            )

        # Datasets para os embeddings
        for nome_modelo, dim_modelo in MODELOS_EMB_NOME_E_DIM_EMB:
            if nome_modelo not in f:
                # Quando criar o dataset com os embeddings do modelo, cria preenchido com nan
                ds = f.create_dataset(
                    nome_modelo,
                    shape=(n, dim_modelo),
                    maxshape=(n, dim_modelo),
                    dtype=np.float16,
                    compression="gzip",
                    chunks=(chunk_size, dim_modelo)
                )
                ds[:] = np.nan

criar_estrutura_embeddings(ARQUIVO_EMBEDDINGS_DOCS, 'DOC_KEY', docs["KEY"].tolist())
criar_estrutura_embeddings(ARQUIVO_EMBEDDINGS_QUERIES, 'QUERY_KEY', queries["KEY"].tolist())

Funções auxiliares para saber se já existe embedding associado e para atualizar embeddings:

In [8]:
def existe_embedding(arquivo, nome_modelo, idx):
    with h5py.File(arquivo, "a") as f:
        ds = f[nome_modelo]

        return not np.isnan(ds[idx, 0])
    
def atualizar_embedding(arquivo, nome_modelo, idx, embedding):
    with h5py.File(arquivo, "a") as f:
        ds = f[nome_modelo]

        ds[idx] = np.asarray(embedding, dtype=np.float16)

## 3. Cria embeddings

Funções auxiliares para geração de embeddings

In [9]:
def reduz_texto_entrada(texto, nome_modelo, exc: Exception):
    error_text = str(exc)

    if nome_modelo == 'text-embedding-3-large' or nome_modelo == 'text-embedding-3-small':
        match = re.search(r'requested\s+(\d+)\s+tokens', exc.message)
        
        total_token_requisitados = int(match.group(1))
        perc_reduzir = 8192/total_token_requisitados * .95
        texto = texto[:int(len(texto) * perc_reduzir)]
        
        return texto

In [10]:
# A id só é usada para fazer o log em caso de erro
def extrai_emb_com_retry(id, texto, nome_modelo, func_get_emb):
    try:
        return func_get_emb(nome_modelo, texto)
    except Exception as e:
        #tqdm.write(f'id: {id}. Texto muito grande. Reduzindo o tamanho')
        return extrai_emb_com_retry(id, reduz_texto_entrada(texto, nome_modelo, e), nome_modelo, func_get_emb)

Função para extrair embeddings para os modelos da openAI:

In [11]:
client_openai = OpenAI(api_key=OPENAI_KEY, base_url=None)
def extrair_embeddings_openai(nome_modelo, texto):
   return client_openai.embeddings.create(input = [texto], model=nome_modelo).data[0].embedding

In [14]:
mapa_func_extrair_embeddings = {
    NOME_MODELO_EMB_OPENAI_LARGE: extrair_embeddings_openai,
    NOME_MODELO_EMB_OPENAI_SMALL: extrair_embeddings_openai,
}

Agora gera os embeddings do texto da norma:

In [15]:
# Varre todos os textos das normas
for idx, row in tqdm(docs.iterrows(), total=len(docs)):
    doc_key = row['KEY']
    # Tem uma norma com mais de 1milhão de caracteres que extrapola até o limite de tokens de entrada por minuto.
    # (mais de 300k, mesmo o limite da janela de entrada sendo muito menor que isso - 8k pros modelos da OpenAI).
    texto = html_to_plain_text(row['TEXTONORMA'])[:1_000_000]
    # Algumas normas tem o texto vazio. Então considera o assunto ou até mesmo a key, apenas para ter um conjunto de emb indexado
    # Por exemplo: NORMA-7114, NORMA-4885, NORMA-4848...
    if texto == '':
        texto = row['ASSUNTO']
        print(f'Texto da norma {doc_key} vazio')
    if texto == '':
        texto = row['KEY']
        print(f'Assunto da norma {doc_key} vazio')

    for nome_modelo, _ in MODELOS_EMB_PARA_GERAR:
        if not existe_embedding(ARQUIVO_EMBEDDINGS_DOCS, nome_modelo, idx):
            func_extrair_emb = mapa_func_extrair_embeddings[nome_modelo]
            emb_gerado = extrai_emb_com_retry(doc_key, texto, nome_modelo, func_extrair_emb)
            atualizar_embedding(ARQUIVO_EMBEDDINGS_DOCS, nome_modelo, idx, emb_gerado)

 54%|█████████████████████████████████████████▊                                   | 7856/14469 [49:56<35:39,  3.09it/s]

Texto da norma NORMA-7114 vazio


 61%|█████████████████████████████████████████████▊                             | 8850/14469 [56:18<1:03:22,  1.48it/s]

Texto da norma NORMA-4885 vazio
Assunto da norma NORMA-4885 vazio


 64%|█████████████████████████████████████████████████▎                           | 9267/14469 [59:11<32:13,  2.69it/s]

...

 86%|███████████████████████████████████████████████████████████████▊          | 12472/14469 [1:22:28<12:40,  2.62it/s]

Texto da norma NORMA-18218 vazio
Assunto da norma NORMA-18218 vazio


 94%|█████████████████████████████████████████████████████████████████████▏    | 13530/14469 [1:30:05<08:03,  1.94it/s]

Texto da norma NORMA-464 vazio


100%|██████████████████████████████████████████████████████████████████████████| 14469/14469 [1:37:43<00:00,  2.47it/s]


Gera os embeddings das queries:

In [16]:
# Varre todos os enunciados das queries
for idx, row in tqdm(queries.iterrows(), total=len(queries)):
    texto = row['TEXT']

    for nome_modelo, _ in MODELOS_EMB_PARA_GERAR:
        if not existe_embedding(ARQUIVO_EMBEDDINGS_QUERIES, nome_modelo, idx):
            func_extrair_emb = mapa_func_extrair_embeddings[nome_modelo]
            emb_gerado = extrai_emb_com_retry(id, texto, nome_modelo, func_extrair_emb)
            atualizar_embedding(ARQUIVO_EMBEDDINGS_QUERIES, nome_modelo, idx, emb_gerado)

100%|██████████████████████████████████████████████████████████████████████████████████| 46/46 [00:14<00:00,  3.10it/s]


## 4. Pesquisa semântica

### 4.1. Funções para carregar e normalizar embeddings

Funções genéricas para carregar e normalizar os embeddings do arquivo h5.

A ideia é que há apenas dois arquivos, um para os embeddings das questões e outro para os embeddings dos chunks.

O:.: Os embeddings da OpenAI já são normalizados (mas há uma perda na conversão de f32 pra f16 para salvar no H5), nem precisaria disso.

In [17]:
def carregar_embeddings_do_arquivo(arquivo, coluna_id, nome_modelo):
    with h5py.File(arquivo, "r") as f:
        ids = f[coluna_id][:].astype(str)
        emb = f[nome_modelo][:].astype(np.float32)

    return ids, emb

def normalizar_embeddings(x):
    faiss.normalize_L2(x)
    return x

### 4.2. Criação de índices FAISS para os modelos

Função para criar um índice FAISS.

In [18]:
def criar_indice_faiss(nome_modelo):
    arquivo_docs = f'{ARQUIVO_EMBEDDINGS_DOCS[:-3]}_{nome_modelo}.h5'
    
    doc_keys, emb = carregar_embeddings_do_arquivo(arquivo_docs, 'DOC_KEY', nome_modelo)
    emb = normalizar_embeddings(emb)
    dim = emb.shape[1]
        
    index_faiss = faiss.IndexFlatL2(dim)
    index_faiss.add(emb)

    return doc_keys, index_faiss

Cria os índices FAISS para cada modelo.

Como serão testados mais de um modelo de embeddings, será criado um mapa de índice no seguinte formato:

<code>lista_doc_key = [....]</code>

<code>mapa_indice_faiss = {
   'nome_modelo_1': indice_faiss,
   'nome_modelo_...': indice_faiss,
   'nome_modelo_n': indice_faiss
}</code>

Obs.: Devido à forma como os embeddings foram criados, todos os DOC_KEY estão na mesma sequência. Por isso a lista de doc_key é separada do mapa.

In [19]:
lista_doc_key = []
mapa_indice_faiss = {}
for nome_modelo, _ in tqdm(MODELOS_EMB_NOME_E_DIM_EMB):
    lista_doc_key, indice = criar_indice_faiss(nome_modelo)
    mapa_indice_faiss[nome_modelo] = indice

100%|████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.87it/s]


### 4.2. Mapa de queries por modelo

Cria mapa dos embeddings das queries por modelo.

Como serão testados mais de um modelo de embeddings, será criado um mapa da seguinte forma:

<code>lista_query_keys = []</code>

<code>mapa_emb_queries = {
   'nome_modelo_1': lista_de_embeddings,
   'nome_modelo_...': lista_de_embeddings,
   'nome_modelo_n: lista_de_embeddings
}

Assim como para o mapa de índice, todos os QUERY_KEY estão na mesma sequência.

In [20]:
lista_query_keys = []
mapa_emb_queries = {}
for nome_modelo, _ in tqdm(MODELOS_EMB_NOME_E_DIM_EMB):
    arquivo_queries = f'{ARQUIVO_EMBEDDINGS_QUERIES[:-3]}_{nome_modelo}.h5'
    lista_query_keys, emb_queries = carregar_embeddings_do_arquivo(arquivo_queries, 'QUERY_KEY', nome_modelo)
    emb_queries = normalizar_embeddings(emb_queries)
    mapa_emb_queries[nome_modelo] = emb_queries

100%|███████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 130.63it/s]


### 4.3. Pesquisa as queries nos índices

Estrutura para guardar o resultado de todos os modelos

In [21]:
df_resultados_pesquisas_semanticas = pd.DataFrame(columns=["MODELO", "QUERY_KEY", "DOC_KEY", "RANK"])

In [22]:
total_porcentagem = len(MODELOS_EMB_NOME_E_DIM_EMB)*len(queries)

def get_doc_key_score(distancias, indices_retornados):
    # Dados para retornar
    top_doc_keys = []
    scores = []

    # As distâncias e os índices retornados são no shape (1, k). Primeiro, transforma tudo em lista de tamanho k:
    distancias = list(distancias[0])
    indices_retornados = indices_retornados = list(indices_retornados[0])
      
    for d, i in zip(distancias, indices_retornados):
        top_doc_keys.append(lista_doc_key[i])
        scores.append(1 - d/2) # O score dessa forma é a similaridade de cosseno
        
    return top_doc_keys, scores

with tqdm(total=total_porcentagem) as pbar:
    lista_resultados_pesquisa_semantica = []
    for nome_modelo, _ in MODELOS_EMB_NOME_E_DIM_EMB:
        embeddings = mapa_emb_queries[nome_modelo]

        for idx, id_query in enumerate(lista_query_keys):
            emb_query = embeddings[idx:idx+1] # shape (1, dim)

            # Consulta no índice de todos os chunks
            distancias, indices_retornados = mapa_indice_faiss[nome_modelo].search(emb_query, 50)
            # Já está na ordem correta
            top_doc_keys, scores = get_doc_key_score(distancias, indices_retornados)
            #resultados_pesquisas_semanticas[id_query][nome_modelo] = { 'doc_key': top_doc_keys, 'scores': scores }
            
            # Adicionar ao df_resultados_pesquisas_semanticas 
            for rank, doc_key in enumerate(top_doc_keys, start=1):
                lista_resultados_pesquisa_semantica.append({
                    "MODELO": nome_modelo,
                    "QUERY_KEY": id_query,
                    "DOC_KEY": doc_key,
                    "RANK": rank
                })

            pbar.update(1)
df_resultados_pesquisas_semanticas = pd.DataFrame(lista_resultados_pesquisa_semantica)
df_resultados_pesquisas_semanticas["QUERY_KEY"] = (df_resultados_pesquisas_semanticas["QUERY_KEY"].astype(int))

100%|█████████████████████████████████████████████████████████████████████████████████| 92/92 [00:00<00:00, 101.25it/s]


### 4.4. Mostra métricas

In [26]:
from metricas import metricas

for nome_modelo, _ in MODELOS_EMB_NOME_E_DIM_EMB:
    # Filtra dos resultados apenas os casos em que MODELO = nome_modelo
    df_resultados_do_modelo = df_resultados_pesquisas_semanticas[df_resultados_pesquisas_semanticas["MODELO"] == nome_modelo]
    df_metricas_qrels = metricas(df_resultados_do_modelo, qrels, aproximacao_trec_eval=True, k=[5,10])
    print(f'nDCG@10: {nome_modelo}')
    print(f'qrel humano: {df_metricas_qrels['nDCG@10'].mean()}')

nDCG@10: text-embedding-3-large
qrel humano: 0.2334670203015551
nDCG@10: text-embedding-3-small
qrel humano: 0.26205167433861243
